# Reward models: hands-on (optional)
**LLMs, Animated · Lesson M3.2**

You'll build the core of a reward model's training in plain NumPy: the Bradley–Terry probability, its loss and gradient, and then a tiny "reward model" that learns one score per response from a few hundred simulated human comparisons.

Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; silence (plus a ✓) means success.

Stuck? Solutions are at the very bottom. Try first, though: *struggling to recall is what makes it stick.*

Only `numpy` and `matplotlib` are needed (both pre-installed on Colab). Runs in well under a minute.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
print("ready")

## 1 · The Bradley–Terry probability
A reward model gives each response a number. Bradley–Terry turns two numbers into the probability that a person prefers the first:

$$P(y_w \succ y_l) = \sigma(r_w - r_l), \qquad \sigma(z) = \frac{1}{1 + e^{-z}}$$

Implement a numerically **stable** sigmoid (a naive `1 / (1 + np.exp(-z))` overflows for very negative `z`) and `bt_prob`.

In [ ]:
def sigmoid(z):
    # TODO: return 1 / (1 + e^(-z)) elementwise, without overflow for large |z|.
    # Hint: 1 / (1 + e^(-z)) = exp(-log(1 + e^(-z))) = np.exp(-np.logaddexp(0, -z))
    raise NotImplementedError("Exercise 1a")

def bt_prob(r_w, r_l):
    # TODO: the Bradley–Terry probability that the chosen response (reward r_w) beats the rejected one (r_l)
    raise NotImplementedError("Exercise 1b")

**Check 1**

In [ ]:
assert np.isclose(sigmoid(0.0), 0.5)
assert np.isclose(bt_prob(2.0, 0.5), 0.8176, atol=1e-4)      # the lesson's worked example
assert np.isclose(bt_prob(3.0, 1.0), bt_prob(13.0, 11.0))     # only the difference matters
with np.errstate(over="raise", invalid="raise"):
    s = sigmoid(np.array([-1000.0, 0.0, 1000.0]))
assert np.allclose(s, [0.0, 0.5, 1.0])
print("✓ exercise 1")

## 2 · The reward-model loss
Training maximises the probability of the choices people actually made. Over a batch of pairs that's the mean negative log-likelihood:

$$\mathcal{L} = -\frac{1}{N}\sum_{k=1}^{N} \log \sigma\big(r_w^{(k)} - r_l^{(k)}\big)$$

In [ ]:
def bt_loss(r_w, r_l):
    # TODO: the mean over pairs of -log σ(r_w - r_l).
    # Hint: -log σ(d) = log(1 + e^(-d)) = np.logaddexp(0, -d), which never overflows.
    raise NotImplementedError("Exercise 2")

**Check 2.** These are the lesson's worked and faded examples.

In [ ]:
assert np.isclose(bt_loss(2.0, 0.5), 0.2014, atol=1e-4)
assert np.isclose(bt_loss(0.3, 1.1), 1.1711, atol=1e-4)
assert np.isclose(bt_loss(1.0, 1.0), np.log(2)), "equal rewards: a coin flip costs log 2"
assert np.isclose(bt_loss(np.array([2.0, 0.3]), np.array([0.5, 1.1])), (0.2014 + 1.1711) / 2, atol=1e-4)
assert np.isfinite(bt_loss(-500.0, 500.0)), "very wrong pairs must not overflow"
print("✓ exercise 2")

## 3 · Its gradient
With $\Delta = r_w - r_l$, the per-pair loss $-\log\sigma(\Delta)$ has

$$\frac{\partial \mathcal{L}}{\partial r_w} = -\big(1 - \sigma(\Delta)\big), \qquad \frac{\partial \mathcal{L}}{\partial r_l} = +\big(1 - \sigma(\Delta)\big)$$

Implement it. The check compares it with a numerical derivative.

In [ ]:
def bt_grads(r_w, r_l):
    # TODO: return (dL/dr_w, dL/dr_l) for the per-pair loss L = -log σ(r_w - r_l)
    # (elementwise if r_w and r_l are arrays)
    raise NotImplementedError("Exercise 3")

**Check 3**

In [ ]:
gw, gl = bt_grads(0.3, 1.1)
assert np.isclose(gw, -0.69, atol=0.01) and np.isclose(gl, 0.69, atol=0.01)   # faded example
eps = 1e-6
numeric = (bt_loss(0.3 + eps, 1.1) - bt_loss(0.3 - eps, 1.1)) / (2 * eps)
assert np.isclose(gw, numeric, atol=1e-6)
gw2, _ = bt_grads(5.0, -5.0)
assert abs(gw2) < 1e-4, "a confidently-right pair should barely push"
print("✓ exercise 3")

## 4 · Learn rewards from comparisons
Six responses have hidden "true" rewards. We simulate 400 human comparisons: each time two different responses are shown, and the better one wins with Bradley–Terry probability, so **upsets happen**, just like real, noisy labels.

Your job: recover the rewards from the comparisons alone, by gradient descent on the loss. A real reward model does exactly this, except that each reward comes out of a transformer instead of a lookup table.

In [ ]:
true_r = np.array([2.0, 1.2, 0.5, 0.0, -0.8, -1.5])
n_items, n_pairs = len(true_r), 400

def simulate(n_pairs, true_r, rng):
    a = rng.integers(0, len(true_r), n_pairs)
    b = (a + rng.integers(1, len(true_r), n_pairs)) % len(true_r)   # always a different item
    a_wins = rng.random(n_pairs) < bt_prob(true_r[a], true_r[b])
    return np.where(a_wins, a, b), np.where(a_wins, b, a)            # (winner, loser) indices

w_idx, l_idx = simulate(n_pairs, true_r, rng)
print("first pairs (winner, loser):", list(zip(w_idx[:5].tolist(), l_idx[:5].tolist())))
print("share of upsets (the worse response won):", np.mean(true_r[w_idx] < true_r[l_idx]).round(2))

In [ ]:
def fit_rewards(w_idx, l_idx, n_items, lr=1.0, steps=500):
    r = np.zeros(n_items)          # one scalar reward per response, all start at 0
    losses = []
    for _ in range(steps):
        # TODO 1: per-pair gradients: gw, gl = bt_grads(r[w_idx], r[l_idx])
        # TODO 2: add them up per item: grad = np.zeros(n_items); np.add.at(grad, w_idx, gw); np.add.at(grad, l_idx, gl)
        #         then divide by the number of pairs (we minimise the MEAN loss)
        # TODO 3: gradient-descent step on r, then subtract r.mean() (only differences matter)
        # TODO 4: losses.append(bt_loss(r[w_idx], r[l_idx]))
        raise NotImplementedError("Exercise 4")
    return r, losses

**Check 4**

In [ ]:
r_hat, losses = fit_rewards(w_idx, l_idx, n_items)
print("fitted rewards     :", r_hat.round(2))
print("true rewards − mean:", (true_r - true_r.mean()).round(2))
assert abs(r_hat.mean()) < 1e-9
assert losses[-1] < losses[0]
assert (np.argsort(-r_hat) == np.argsort(-true_r)).all(), "the fitted rewards should rank responses like the true ones"
assert np.corrcoef(r_hat, true_r)[0, 1] > 0.95
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(losses); ax[0].set_xlabel("step"); ax[0].set_ylabel("Bradley–Terry loss"); ax[0].set_title("Training loss")
ax[1].scatter(true_r - true_r.mean(), r_hat); ax[1].plot([-2, 2], [-2, 2], "k--", lw=0.8)
ax[1].set_xlabel("true reward (centred)"); ax[1].set_ylabel("fitted reward"); ax[1].set_title("Recovered rewards")
plt.tight_layout(); plt.show()
print("✓ exercise 4")

## Experiment
1. **How much data?** Run the cell below: it refits with 20, 100 and 2,000 comparisons. How does the scatter tighten?
2. **Shift invariance.** `bt_loss(r_hat[w_idx] + 5, r_hat[l_idx] + 5)` vs `bt_loss(r_hat[w_idx], r_hat[l_idx])`: why are they equal? What does that say about reading a reward model's raw score?
3. **Noisier humans.** Replace `true_r` by `0.3 * true_r` in the simulation (preferences closer to coin flips). How many pairs do you now need to get the ranking right?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, n in zip(axes, [20, 100, 2000]):
    w, l = simulate(n, true_r, np.random.default_rng(1))
    r_fit, _ = fit_rewards(w, l, n_items)
    ax.scatter(true_r - true_r.mean(), r_fit); ax.plot([-2, 2], [-2, 2], "k--", lw=0.8)
    ax.set_title(f"{n} comparisons"); ax.set_xlabel("true reward (centred)")
axes[0].set_ylabel("fitted reward")
plt.tight_layout(); plt.show()
print("shifted loss == original loss:", np.isclose(bt_loss(r_hat[w_idx] + 5, r_hat[l_idx] + 5), bt_loss(r_hat[w_idx], r_hat[l_idx])))

## Recap
- Bradley–Terry: $P(y_w \succ y_l) = \sigma(r_w - r_l)$. Only **differences** of rewards matter.
- The reward-model loss is $-\log\sigma(r_w - r_l)$, a binary cross-entropy whose label is always "chosen wins".
- Its gradient pushes the chosen reward up and the rejected one down, hardest when the model is wrong.
- From noisy comparisons alone you can recover a good **ranking**; more pairs give tighter estimates.

Back to the course and hit **Mark complete** to add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def sigmoid(z):
    return np.exp(-np.logaddexp(0, -np.asarray(z, dtype=float)))

def bt_prob(r_w, r_l):
    return sigmoid(np.asarray(r_w) - np.asarray(r_l))

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def bt_loss(r_w, r_l):
    d = np.asarray(r_w, dtype=float) - np.asarray(r_l, dtype=float)
    return float(np.mean(np.logaddexp(0, -d)))

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def bt_grads(r_w, r_l):
    p = sigmoid(np.asarray(r_w, dtype=float) - np.asarray(r_l, dtype=float))
    return -(1 - p), (1 - p)

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def fit_rewards(w_idx, l_idx, n_items, lr=1.0, steps=500):
    r = np.zeros(n_items)
    losses = []
    for _ in range(steps):
        gw, gl = bt_grads(r[w_idx], r[l_idx])
        grad = np.zeros(n_items)
        np.add.at(grad, w_idx, gw)
        np.add.at(grad, l_idx, gl)
        grad /= len(w_idx)
        r = r - lr * grad
        r = r - r.mean()
        losses.append(bt_loss(r[w_idx], r[l_idx]))
    return r, losses